# Self-Adaptive ARM Simulator — API

Este notebook permite **monitorar** e **controlar** o simulador em tempo real via HTTP.

## Pré-requisito
O `managing` precisa estar rodando antes de executar qualquer célula:
```bash
cd managing
python main.py
```

## Endpoints disponíveis
| Método | Endpoint                    | Descrição                                      |
|--------|-----------------------------|------------------------------------------------|
| GET    | `/perception`               | Retorna o último estado do simulador           |
| GET    | `/environment/obstacles`    | Lista obstáculos e suas posições atuais        |
| GET    | `/environment/objects`      | Lista objetos manipuláveis e suas posições     |
| PUT    | `/task`                     | Envia uma task para o braço executar           |
| PUT    | `/waypoints`                | Envia waypoints para a `API_TASK` executar     |
| PUT    | `/environment`              | Adiciona, remove ou move obstáculos e objetos  |
| PUT    | `/goal`                     | Move esferas de goal em tempo de execução      |

Documentação interativa: http://localhost:8000/docs

In [2]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup

In [6]:
%pip install requests

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.4.7-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.5.20-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.4.7-cp313-cp313-win_amd64.whl (158 kB)
Using cached idna-3.18-py3-none-any.whl (65 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
Using cached certifi-2026.5.20-py3-none-any.whl (134 kB)

   -------- ------------------------------- 1/5 [idna]
   -------------------------------- ------- 4/5 [requests]
   ---------------------------------------- 5/5 [requests]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
import requests
import json
import time

BASE_URL = "http://localhost:8000"

def get(endpoint: str) -> dict:
    r = requests.get(f"{BASE_URL}{endpoint}", timeout=3)
    r.raise_for_status()
    return r.json()

def put(endpoint: str, body: dict) -> dict:
    r = requests.put(f"{BASE_URL}{endpoint}", json=body, timeout=3)
    r.raise_for_status()
    return r.json()

def pp(data: dict):
    """Pretty-print de dicionários."""
    print(json.dumps(data, indent=2, default=str))

print("Setup OK")

Setup OK


---
## GET `/perception`
Retorna o snapshot mais recente do estado do simulador.
Pode ser chamado a qualquer momento — devolve sempre o último valor disponível.

In [47]:
perception = get("/perception")
pp(perception)

{
  "episode": 1,
  "step": 22,
  "ee_position": [
    0.23843592405319214,
    0.0011596685508266091,
    0.19739921391010284
  ],
  "ee_velocity": [
    1.868700820750746e-07,
    -1.228795008501038e-05,
    4.5145108629185415e-07
  ],
  "fingers_width": 1.559217643887223e-08,
  "cube_position": [
    0.029998881742358208,
    9.838790901994798e-06,
    0.019989412277936935
  ],
  "target_position": [
    0.15000000596046448,
    0.0,
    0.019999999552965164
  ],
  "dist_ee_to_cube": 0.2737180292606354,
  "dist_cube_to_target": 0.12000112235546112,
  "reward": -0.12000112235546112,
  "is_success": false,
  "obstacle_in_path": true,
  "obstacle_count_in_path": 1,
  "obstacles": {
    "parede_1": {
      "type": "box",
      "size": [
        0.02,
        0.3,
        0.08
      ],
      "mass": 0.0,
      "color": [
        0.8,
        0.2,
        0.2,
        0.9
      ],
      "lateral_friction": null,
      "spinning_friction": null
    }
  },
  "objects": {
    "cube_1": {
   

In [48]:
from IPython.display import clear_output

try:
    while True:
        clear_output(wait=True)
        pp(get("/perception"))
        time.sleep(0.5)
except KeyboardInterrupt:
    print("Loop encerrado.")

{
  "episode": 1,
  "step": 80,
  "ee_position": [
    0.23842890560626984,
    0.0010657885577529669,
    0.19739404320716858
  ],
  "ee_velocity": [
    6.467237767537881e-07,
    -1.0462950740475208e-05,
    9.231505941897922e-07
  ],
  "fingers_width": 1.6795857149531912e-08,
  "cube_position": [
    0.02999788336455822,
    3.5978304367745295e-05,
    0.019989412277936935
  ],
  "target_position": [
    0.15000000596046448,
    0.0,
    0.019999999552965164
  ],
  "dist_ee_to_cube": 0.27370962500572205,
  "dist_cube_to_target": 0.1200021281838417,
  "reward": -0.1200021281838417,
  "is_success": false,
  "obstacle_in_path": true,
  "obstacle_count_in_path": 1,
  "obstacles": {
    "parede_1": {
      "type": "box",
      "size": [
        0.02,
        0.3,
        0.08
      ],
      "mass": 0.0,
      "color": [
        0.8,
        0.2,
        0.2,
        0.9
      ],
      "lateral_friction": null,
      "spinning_friction": null
    }
  },
  "objects": {
    "cube_1": {
   

### Campos principais do perception

In [ ]:
p = get("/perception")

print(f"Episódio / Step    : {p.get('episode')} / {p.get('step')}")
print(f"Task atual         : {p.get('current_task')}")
print(f"Action             : {p.get('action')}")
print()
print(f"EE posição         : {p.get('ee_position')}")
print(f"Cubo posição       : {p.get('cube_position')}")
print(f"Target             : {p.get('target_position')}")
print()
print(f"Dist EE → Cubo     : {p.get('dist_ee_to_cube'):.4f} m")
print(f"Dist Cubo → Target : {p.get('dist_cube_to_target'):.4f} m")
print(f"Reward             : {p.get('reward'):.4f}")
print(f"Sucesso            : {p.get('is_success')}")
print()
print(f"Obstáculo no caminho : {p.get('obstacle_in_path')}  (count: {p.get('obstacle_count_in_path')})")

---
## GET `/environment/obstacles` e `/environment/objects`

Retorna os corpos presentes na cena com suas posições atuais.
O estado é atualizado sempre que um comando `PUT /environment` é processado.

In [68]:
# Lista todos os obstáculos e suas posições atuais
obstacles = get("/environment/obstacles")
pp(obstacles)

# Exemplo de retorno:
# {
#   "parede_1": [0.09, 0.0, 0.04]
# }

{
  "parede_1": {
    "type": "box",
    "size": [
      0.02,
      0.3,
      0.08
    ],
    "mass": 0.0,
    "color": [
      0.8,
      0.2,
      0.2,
      0.9
    ],
    "current_position": [
      0.0,
      0.0,
      0.04
    ]
  }
}


In [ ]:
# Lista todos os objetos manipuláveis e suas posições atuais
objects = get("/environment/objects")
pp(objects)

# Exemplo de retorno:
# {
#   "object_1": {"type": "box", "size": [...], "current_position": [0.03, 0.0, 0.02], ...}
# }

---
## PUT `/task`
Envia uma task para o braço executar.

### Tasks disponíveis
| Valor                      | Descrição                                      |
|----------------------------|------------------------------------------------|
| `PUSH`                     | Empurra o cubo em direção ao target            |
| `PICK_AND_PLACE`           | Pega o cubo e coloca no target                 |
| `REACH`                    | Move o end-effector até o target               |
| `HOLD`                     | Segura o cubo na posição atual                 |
| `MANUAL`                   | Controle manual via teclado (janela PyBullet)  |
| `API_TASK`                 | Waypoints enviados via `PUT /waypoints`        |
| `SCRIPTED_TASK.script_1`   | Executa o script `script_1` (scripts.yaml)     |
| `SCRIPTED_TASK.reach_only` | Executa o script `reach_only`                  |
| `SCRIPTED_TASK.left_right` | Executa o script `left_right`                  |

In [103]:
# Enviar PUSH
response = put("/task", {"strategy": "HOLD"})
print(response)

{'ok': True}


In [100]:
# Enviar PUSH
response = put("/task", {"strategy": "PUSH"})
print(response)

{'ok': True}


In [126]:
# Enviar PICK_AND_PLACE
response = put("/task", {"strategy": "PICK_AND_PLACE"})
print(response)

{'ok': True}


In [102]:
# Enviar script
response = put("/task", {"strategy": "SCRIPTED_TASK.left_right"})
print(response)

{'ok': True}


---
## PUT `/waypoints` — APITask
Controla o braço enviando posições diretamente via API.

**Passo 1** — ativar a `API_TASK`:
```python
put("/task", {"strategy": "API_TASK"})
```

**Passo 2** — enviar waypoints via `PUT /waypoints`.

### Formato
| Tipo | Body |
|---|---|
| Waypoint único | `{"waypoints": [x, y, z, gripper]}` |
| Sequência | `{"waypoints": [[x,y,z,g], [x,y,z,g], ...]}` |

`gripper`: `1.0` = aberta · `-1.0` = fechada

In [ ]:
# Passo 1 — ativar APITask
put("/task", {"strategy": "API_TASK"})
print("APITask ativada.")

# Waypoint único — [x, y, z, gripper]
# gripper: 1.0 = aberta, -1.0 = fechada
response = put("/waypoints", {"waypoints": [0.1, 0.0, 0.3, 1.0]})
print(response)

{'ok': True}


In [ ]:
# Passo 1 — ativar APITask
put("/task", {"strategy": "API_TASK"})
print("APITask ativada.")

# Sequência de waypoints — [[x, y, z, gripper], ...]
# O braço executa cada waypoint em ordem, avançando quando chega perto o suficiente
response = put("/waypoints", {"waypoints": [
    [ 0.1,  0.0,  0.4,  1.0],   # mover para cima com garra aberta
    [ 0.1,  0.0,  0.2,  1.0],   # descer
    [ 0.1,  0.0,  0.2, -1.0],   # fechar garra
    [ 0.1,  0.0,  0.4, -1.0],   # subir com garra fechada
]})
print(response)

{'ok': True}


---
## Monitoramento contínuo
Faz polling do `/perception` a cada segundo e exibe os campos principais.

> **Parar**: interrompa o kernel (`■` na barra do Jupyter).

In [ ]:
from IPython.display import clear_output

INTERVAL = 1.0  # segundos entre cada leitura

try:
    while True:
        p = get("/perception")
        clear_output(wait=True)

        print(f"Ep {p.get('episode'):>2} | Step {p.get('step'):>3}")
        print(f"Task          : {p.get('current_task')}")
        print(f"Action        : {p.get('action')}")
        print(f"EE posição    : {p.get('ee_position')}")
        print(f"Cubo posição  : {p.get('cube_position')}")
        print(f"Dist → target : {p.get('dist_cube_to_target'):.4f} m")
        print(f"Reward        : {p.get('reward'):+.4f}")
        print(f"Sucesso       : {p.get('is_success')}")
        print(f"Obstáculo     : {p.get('obstacle_in_path')}  (count: {p.get('obstacle_count_in_path')})")

        time.sleep(INTERVAL)
except KeyboardInterrupt:
    print("Monitoramento encerrado.")

---
## Sequência de tasks
Envia tasks em sequência com intervalo entre elas — útil para testar trocas de comportamento.

In [ ]:
sequence = [
    ("PUSH",                      5),   # PUSH por 5 segundos
    ("PICK_AND_PLACE",            5),   # PICK_AND_PLACE por 5 segundos
    ("SCRIPTED_TASK.left_right",  5),   # script left_right por 5 segundos
]

for strategy, duration in sequence:
    print(f"→ Enviando: {strategy}")
    put("/task", {"strategy": strategy})
    time.sleep(duration)

print("Sequência concluída.")

---
## PUT `/environment` — Alterações de ambiente em tempo de execução

Permite modificar a cena física enquanto a simulação está rodando.
A mudança é aplicada imediatamente no próximo step do simulador.

### Ações disponíveis

| `action`           | Descrição                              | Campos obrigatórios                          |
|--------------------|----------------------------------------|----------------------------------------------|
| `move_obstacle`    | Move um obstáculo existente            | `name`, `position`                           |
| `add_obstacle`     | Adiciona um novo obstáculo na cena     | `name`, `size`, `position`, `color`          |
| `remove_obstacle`  | Remove um obstáculo da cena            | `name`                                       |
| `move_object`      | Teleporta um objeto manipulável        | `name`, `position`                           |
| `move_robot_base`  | Reposiciona a base do braço robótico   | `position`                                   |

> **`position`**: `[x, y, z]` em metros no referencial do mundo (z=0 = superfície da mesa)  
> **`size`**: `[x, y, z]` tamanho total em metros (não half-extents)  
> **`color`**: `[r, g, b, a]` com valores entre 0.0 e 1.0

### `move_obstacle` — mover um obstáculo existente
Move o `obstacle_1` (ou qualquer obstáculo pelo nome) para uma nova posição.
O obstáculo precisa já existir na cena — use `add_obstacle` caso contrário.

In [89]:
# move_obstacle — reposiciona obstacle_1 para uma nova posição
response = put("/environment", {
    "action":   "move_obstacle",
    "name":     "obstacle_1",
    "position": [0.09, 0.0, 0.04]   # [x, y, z]
})
print(response)

{'ok': True}


### `add_obstacle` — adicionar um novo obstáculo
Cria um novo obstáculo na cena em tempo de execução.
O `name` deve ser único — não pode conflitar com nomes já existentes (`obstacle_1`, `object_1`, `table`, etc).

In [109]:
# add_obstacle — cria uma nova parede lateral na cena
response = put("/environment", {
    "action":   "add_obstacle",
    "name":     "obstacle_2",
    "size":     [0.02, 0.30, 0.08],   # [largura_x, profundidade_y, altura_z]
    "position": [0.09, 0.15, 0.04],   # [x, y, z] — centro do obstáculo
    "color":    [0.8, 0.5, 0.1, 0.9], # [r, g, b, a]
    "mass":     0.0                    # 0.0 = estático (não cai)
})
print(response)

{'ok': True}


### `remove_obstacle` — remover um obstáculo
Remove permanentemente um obstáculo da cena.
Após remover, o nome fica disponível para ser reusado em um `add_obstacle`.

In [110]:
# remove_obstacle — remove obstacle_1 da cena
response = put("/environment", {
    "action": "remove_obstacle",
    "name":   "obstacle_2"
})
print(response)

{'ok': True}


### `move_object` — teleportar o cubo
Reposiciona o objeto manipulável (cubo) instantaneamente.
Útil para resetar a posição do cubo sem reiniciar o episódio.

In [107]:
# move_object — reposiciona object_1 para a posição inicial
response = put("/environment", {
    "action":   "move_object",
    "name":     "object_1",
    "position": [0.03, 0.1, 0.09]   # posição inicial padrão do objeto
})
print(response)

{'ok': True}


### `move_robot_base` — reposicionar a base do braço
Move a base do robô para uma nova posição em tempo de execução.
O controle EE usa IK em coordenadas do mundo — o braço se adapta automaticamente à nova base no próximo step.

> **Atenção**: se mover a base enquanto o braço está nas fases `LIFT` ou `RELEASE` do pick-and-place,
> ele tentará alcançar o target fixado antes do movimento — pode não conseguir se a nova base estiver muito longe.

In [ ]:
# move_robot_base — reposiciona a base do braço para uma nova posição
# Posição padrão: [-0.4, 0.0, 0.0]
response = put("/environment", {
    "action":   "move_robot_base",
    "position": [-0.2, 0.0, 0.0]   # mover base 20 cm para frente
})
print(response)

In [ ]:


# Restaurar posição original
response = put("/environment", {
    "action":   "move_robot_base",
    "position": [-0.4, 0.0, 0.0]
})
print(response)

---
## PUT `/goal` — Mover esferas de goal em tempo de execução

Move as esferas de objetivo visíveis na cena.
Útil para mudar o destino do braço sem reiniciar o episódio.

### Ação disponível

| `action`       | Descrição                        | Campos obrigatórios  |
|----------------|----------------------------------|----------------------|
| `move_target`  | Move uma esfera de goal          | `name`, `position`   |

### Nomes das esferas
| Nome       | Descrição                          |
|------------|------------------------------------|
| `target`   | Primeiro goal (ou único)           |
| `target_1` | Segundo goal (se houver sequência) |
| `target_2` | Terceiro goal                      |
| `target_N` | N+1-ésimo goal                     |

> A mudança é visual e física — o braço passará a usar a nova posição como destino no próximo step.

### `move_target` — mover o goal principal

In [117]:
# move_target — move o goal principal para uma nova posição
response = put("/goal", {
    "action":   "move_target",
    "name":     "target_3",
    "position": [0.20, 0.0, 0.02]   # [x, y, z]
})
print(response)

{'ok': True}


In [ ]:
# move_target — mover goals de uma sequência individualmente
# target   = primeiro goal
# target_1 = segundo goal
# target_2 = terceiro goal

put("/goal", {"action": "move_target", "name": "target",   "position": [0.15,  0.0,  0.02]})
put("/goal", {"action": "move_target", "name": "target_1", "position": [0.15,  0.10, 0.02]})
put("/goal", {"action": "move_target", "name": "target_2", "position": [-0.10, 0.0,  0.02]})
print("Goals reposicionados.")

### `set_goal_mode` — trocar o modo de seleção de goal em tempo de execução

Muda como o braço escolhe qual goal perseguir, sem reiniciar o episódio.

| `mode`            | Comportamento                                              |
|-------------------|------------------------------------------------------------|
| `goal_options`    | Vai sempre para o target **mais próximo** da posição atual |
| `goal_sequence`   | Segue os targets **em ordem**, um por vez                  |
| `goal_set`        | Alcança **todos** os targets, escolhendo o mais próximo    |

> O reset interno (índice e completados) acontece automaticamente ao trocar o modo.

In [127]:
# set_goal_mode — trocar para "mais próximo de todos" (goal_options)
response = put("/goal", {
    "action": "set_goal_mode",
    "mode":   "goal_options"
})
print(response)

{'ok': True}


In [128]:
# Enviar PICK_AND_PLACE
response = put("/task", {"strategy": "PICK_AND_PLACE"})
print(response)

{'ok': True}


In [ ]:


# set_goal_mode — voltar para sequência fixa (goal_sequence)
response = put("/goal", {
    "action": "set_goal_mode",
    "mode":   "goal_sequence"
})
print(response)

# set_goal_mode — alcançar todos, em qualquer ordem (goal_set)
response = put("/goal", {
    "action": "set_goal_mode",
    "mode":   "goal_set"
})
print(response)